In [2]:
import pandas as pd

df_r = pd.read_csv("../Data/cleaned/readmissions_and_deaths_cleaned.csv")
print("Shape:", df_r.shape)
print("\nColumns:")
for col in df_r.columns:
    print(f"  {col}")

Shape: (95780, 17)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  measure_id
  measure_name
  compared_to_national
  denominator
  score
  lower_estimate
  higher_estimate
  start_date
  end_date


In [3]:
print("unique measures:", df_r["measure_id"].nunique())
print("\nmeasure list:")
print(df_r[["measure_id", "measure_name"]].drop_duplicates().to_string())

unique measures: 20

measure list:
       measure_id                                                               measure_name
0   COMP_HIP_KNEE                    Rate of complications for hip/knee replacement patients
1      Hybrid_HWM            Hybrid Hospital-Wide All-Cause Risk Standardized Mortality Rate
2     MORT_30_AMI                                       Death rate for heart attack patients
3    MORT_30_CABG                                       Death rate for CABG surgery patients
4    MORT_30_COPD                                               Death rate for COPD patients
5      MORT_30_HF                                      Death rate for heart failure patients
6      MORT_30_PN                                          Death rate for pneumonia patients
7     MORT_30_STK                                             Death rate for stroke patients
8          PSI_03                                                        Pressure ulcer rate
9          PSI_04  Death rate among

In [4]:
print("compared_to_national values:")
print(df_r["compared_to_national"].value_counts(dropna=False))

print("\nscore sample:")
print(df_r["score"].value_counts().head(10))

compared_to_national values:
compared_to_national
No Different Than the National Rate     46793
Not Available                           31720
Number of Cases Too Small               11926
No Different Than the National Value     2697
Better Than the National Rate            1296
Worse Than the National Rate             1125
Worse Than the National Value             162
Better Than the National Value             61
Name: count, dtype: int64

score sample:
score
Not Available    43646
0.27               781
0.21               736
0.20               657
0.26               654
0.25               464
4.1                453
4                  430
4.2                399
4.3                376
Name: count, dtype: int64


In [5]:
df_r["score"] = df_r["score"].replace("Not Available", None)
df_r["score"] = pd.to_numeric(df_r["score"])

df_r["compared_to_national"] = df_r["compared_to_national"].replace({
    "Not Available": None,
    "No Different Than the National Value": "No Different Than the National Rate",
    "Better Than the National Value": "Better Than the National Rate",
    "Worse Than the National Value": "Worse Than the National Rate"
})

print("score nulls:", df_r["score"].isnull().sum())
print("\ncompared_to_national values:")
print(df_r["compared_to_national"].value_counts(dropna=False))

score nulls: 43646

compared_to_national values:
compared_to_national
No Different Than the National Rate    49490
None                                   31720
Number of Cases Too Small              11926
Better Than the National Rate           1357
Worse Than the National Rate            1287
Name: count, dtype: int64


In [6]:
df_r["compared_to_national"] = df_r["compared_to_national"].replace("None", None)

print(df_r["compared_to_national"].value_counts(dropna=False))

compared_to_national
No Different Than the National Rate    49490
None                                   31720
Number of Cases Too Small              11926
Better Than the National Rate           1357
Worse Than the National Rate            1287
Name: count, dtype: int64


In [7]:
import numpy as np

df_r["compared_to_national"] = df_r["compared_to_national"].replace("None", np.nan)

print(df_r["compared_to_national"].value_counts(dropna=False))

compared_to_national
No Different Than the National Rate    49490
None                                   31720
Number of Cases Too Small              11926
Better Than the National Rate           1357
Worse Than the National Rate            1287
Name: count, dtype: int64


In [8]:
df_r.loc[df_r["compared_to_national"] == "None", "compared_to_national"] = np.nan

print(df_r["compared_to_national"].value_counts(dropna=False))

compared_to_national
No Different Than the National Rate    49490
None                                   31720
Number of Cases Too Small              11926
Better Than the National Rate           1357
Worse Than the National Rate            1287
Name: count, dtype: int64


In [9]:
sample = df_r["compared_to_national"].value_counts(dropna=False).index.tolist()
for val in sample:
    print(repr(val))

'No Different Than the National Rate'
None
'Number of Cases Too Small'
'Better Than the National Rate'
'Worse Than the National Rate'


In [10]:
print("actual null count:", df_r["compared_to_national"].isnull().sum())


actual null count: 31720


In [11]:
score_pivot = df_r.pivot_table(
    index="facility_id",
    columns="measure_id",
    values="score",
    aggfunc="first"
).reset_index()

# flatten column names
score_pivot.columns.name = None

print("shape after pivot:", score_pivot.shape)
print(score_pivot.head(2))

shape after pivot: (4184, 21)
  facility_id  COMP_HIP_KNEE  Hybrid_HWM  MORT_30_AMI  MORT_30_CABG  \
0      010001            3.2         4.5         11.4           3.0   
1      010005            3.0         4.6          NaN           NaN   

   MORT_30_COPD  MORT_30_HF  MORT_30_PN  MORT_30_STK  PSI_03  ...  PSI_06  \
0           9.4        10.2        18.4         13.5    0.13  ...    0.13   
1           8.9        14.1        21.2         12.9    0.33  ...    0.19   

   PSI_08  PSI_09  PSI_10  PSI_11  PSI_12  PSI_13  PSI_14  PSI_15  PSI_90  
0    0.34    2.75    1.32   10.66    4.40    5.40    1.58    2.09    0.95  
1    0.25    2.04    1.44    8.65    4.89    6.38    1.73    1.23    0.97  

[2 rows x 21 columns]


In [12]:
compared_pivot = df_r.pivot_table(
    index="facility_id",
    columns="measure_id",
    values="compared_to_national",
    aggfunc="first"
).reset_index()

compared_pivot.columns.name = None
compared_pivot.columns = ["facility_id"] + [
    f"{col}_compared" for col in compared_pivot.columns[1:]
]

print("shape:", compared_pivot.shape)
print(compared_pivot.head(2))

shape: (4480, 21)
  facility_id               COMP_HIP_KNEE_compared  \
0      010001  No Different Than the National Rate   
1      010005  No Different Than the National Rate   

                   Hybrid_HWM_compared                 MORT_30_AMI_compared  \
0  No Different Than the National Rate  No Different Than the National Rate   
1  No Different Than the National Rate            Number of Cases Too Small   

                 MORT_30_CABG_compared                MORT_30_COPD_compared  \
0  No Different Than the National Rate  No Different Than the National Rate   
1                                  NaN  No Different Than the National Rate   

                   MORT_30_HF_compared                  MORT_30_PN_compared  \
0  No Different Than the National Rate  No Different Than the National Rate   
1  No Different Than the National Rate         Worse Than the National Rate   

                  MORT_30_STK_compared                      PSI_03_compared  \
0  No Different Than the N

In [13]:
hospital_info = df_r[["facility_id", "facility_name", "address", "citytown", 
                       "state", "zip_code", "countyparish", "telephone_number"]].drop_duplicates("facility_id")

df_final = hospital_info.merge(score_pivot, on="facility_id", how="outer")
df_final = df_final.merge(compared_pivot, on="facility_id", how="outer")

print("final shape:", df_final.shape)
print("columns:", df_final.shape[1])
df_final.head(2)

final shape: (4789, 48)
columns: 48


,facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,COMP_HIP_KNEE,Hybrid_HWM,...,PSI_06_compared,PSI_08_compared,PSI_09_compared,PSI_10_compared,PSI_11_compared,PSI_12_compared,PSI_13_compared,PSI_14_compared,PSI_15_compared,PSI_90_compared
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,3.2,4.5,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,Worse Than the National Rate,No Different Than the National Rate
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,3.0,4.6,...,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate,No Different Than the National Rate


In [14]:
print("duplicate facility_ids:", df_final["facility_id"].duplicated().sum())


df_final.to_csv("../Data/cleaned/readmissions_and_deaths_cleaned.csv", index=False)
print("readmissions_and_deaths saved")

duplicate facility_ids: 0
readmissions_and_deaths saved
